# 99-100 - External Perplexity: L1-only vs Dual (C4 samples 99-199)

**Experiment**: take 100 C4 prompts starting at row 99 (`prompt_text` = first 50 tokens),
generate **200-token** continuations in three variants -

| variant | layers on |
|---|---|
| `plain` | none |
| `l1_only` | public topic greenlist boost (delta_public) |
| `dual` | topic boost + private KGW boost |

then score the **continuation-only perplexity under an independent judge**
(`Qwen/Qwen1.5-7B`, 4-bit so it fits a T4) using `src/metrics/perplexity.py`.

**T4 budget**: ~300 generations (1.5-2 h) + ~15 min Qwen scoring. Everything chunk-saves
to Drive and resumes by id, so a disconnect only loses the current chunk.
Phased on purpose: OPT and Qwen cannot share 16 GB VRAM.

In [3]:
!git clone https://github.com/pravaspaudel/Dual_watermarking_Scheme.git
%cd Dual_watermarking_Scheme
!pip install -q transformers accelerate bitsandbytes scipy pandas matplotlib

Cloning into 'Dual_watermarking_Scheme'...
remote: Enumerating objects: 264, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 264 (delta 56), reused 70 (delta 20), pack-reused 151 (from 1)
Receiving objects: 100% (264/264), 43.59 MiB | 17.98 MiB/s, done.
Resolving deltas: 100% (115/115), done.
/content/Dual_watermarking_Scheme
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.7 MB/s eta 0:00:00:00:0100:01


In [4]:
import gc
import os
import time

import numpy as np
import pandas as pd
import torch

from google.colab import drive
drive.mount('/content/drive')

from src.utils.model import load_model
from src.watermark.dual_layer import DualWaterMarking
from src.metrics.perplexity import (
    load_judge_model,
    split_continuation,
    compute_external_perplexity,
)

# ----------------------------- CONFIG ---------------------------------------
ROW_START      = 99          # first C4 sample (positional iloc)
N_SAMPLES      = 100         # rows 99..198 inclusive
MAX_NEW_TOKENS = 200
DELTA_PUBLIC   = 2.0
DELTA_PRIVATE  = 0.7
JUDGE_NAME     = "Qwen/Qwen1.5-7B"
LOAD_4BIT      = True        # REQUIRED on T4: fp16 weights ~15.4GB > 15GB VRAM
SEED_BASE      = 7000        # per-row seed = SEED_BASE + row id
CHUNK_SAVE     = 5           # flush to Drive every N prompts
VARIANTS       = ["plain", "l1_only", "dual"]
# -----------------------------------------------------------------------------

DRIVE  = "/content/drive/MyDrive/minor_project"
OUTDIR = f"{DRIVE}/eval_results/ppl_99_200"
FIGDIR = f"{DRIVE}/eval_results/figures"
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(FIGDIR, exist_ok=True)
GEN_CSV    = f"{OUTDIR}/gen_99_200.csv"
SCORES_CSV = f"{OUTDIR}/ppl_scores.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
assert DEVICE == "cuda", "Enable GPU runtime: Runtime > Change runtime type > T4 GPU" 

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda


## Phase A - generation (OPT-2.7b on GPU)

Same three generations as `DualWaterMarking.watermark()` minus `layer2_only`
(not needed here), saving ~25% of the generation budget.

In [5]:
model, tokenizer, VOCAB_SIZE = load_model("facebook/opt-2.7b")
model.to(DEVICE).eval()

wm = DualWaterMarking(
    model,
    tokenizer,
    greenlist_dir="data/greenlist",
    split="all",
    delta_public=DELTA_PUBLIC,
    delta_private=DELTA_PRIVATE,
    max_new_tokens=MAX_NEW_TOKENS,
    seed=0,
)
print("key fingerprint:", wm.key[:8], "| topics:", wm.topics)


@torch.no_grad()
def generate_row(idx, prompt):
    torch.manual_seed(SEED_BASE + idx)          # reproducible per-row sampling
    topic, ranked = wm.extract_topic(prompt)
    inputs = wm._inputs(prompt)
    return {
        "id": idx,
        "prompt_text": prompt,
        "topic": topic,
        "topic_score": round(ranked[0][1], 4),
        "plain_output":    wm._generate(inputs),
        "l1_only_output":  wm._generate(inputs, wm._make_processor(topic, 1.0, 0.0)),
        "dual_output":     wm._generate(inputs, wm._make_processor(topic, 1.0, 1.0)),
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.30GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

model and tokenizer of facebook/opt-2.7b loaded with vocab_size 50265
key fingerprint: d96e420b | topics: ['entertainment', 'finance', 'history', 'medicine', 'politics', 'science', 'sports', 'technology']


In [ ]:
from tqdm.auto import tqdm

c4_df = pd.read_csv("data/extracted/c4_samples.csv")
targets = c4_df.iloc[ROW_START:ROW_START + N_SAMPLES].reset_index(drop=True)
print(f"scoring rows {ROW_START}..{ROW_START + N_SAMPLES - 1} ({len(targets)} prompts)")

done_ids = set()
if os.path.exists(GEN_CSV):
    done_ids = set(pd.read_csv(GEN_CSV)["id"].tolist())
    print(f"resume: {len(done_ids)} ids already generated")

rows_buf, t_gen0 = [], time.time()
for pos, row in enumerate(tqdm(targets.itertuples(index=False), total=len(targets))):
    idx = ROW_START + pos
    if idx in done_ids:
        continue
    rows_buf.append(generate_row(idx, row.prompt_text))
    if len(rows_buf) >= CHUNK_SAVE:
        old = pd.read_csv(GEN_CSV) if os.path.exists(GEN_CSV) else None
        merged = pd.concat([old, pd.DataFrame(rows_buf)]) if old is not None else pd.DataFrame(rows_buf)
        save_chunk(merged, GEN_CSV)
        rows_buf = []
if rows_buf:
    old = pd.read_csv(GEN_CSV) if os.path.exists(GEN_CSV) else None
    merged = pd.concat([old, pd.DataFrame(rows_buf)]) if old is not None else pd.DataFrame(rows_buf)
    save_chunk(merged, GEN_CSV)

gen_df = pd.read_csv(GEN_CSV)
t_gen = time.time() - t_gen0
print(f"generation done: {len(gen_df)} rows in {t_gen/60:.1f} min "
      f"({t_gen / max(len(gen_df), 1):.1f} s/prompt incl. 3 variants)")
assert len(gen_df) == N_SAMPLES, "missing rows - rerun this cell to resume" 

## Phase B - free OPT, load Qwen1.5-7B judge (4-bit)

OPT fp32 (~11 GB) + Qwen (~15 GB even in fp16) can never co-exist on a T4,
so everything OPT-related is deleted first. 4-bit puts the judge at ~6 GB
with headroom for activations.

In [ ]:
# None-assign first so this cell is safe to re-run after a restart
wm, model, tokenizer = None, None, None
del wm, model, tokenizer
gc.collect()
torch.cuda.empty_cache()

judge_model, judge_tokenizer = load_judge_model(
    JUDGE_NAME,
    device=DEVICE,
    torch_dtype=torch.float16,
    load_in_4bit=LOAD_4BIT,
)
print("judge loaded:", JUDGE_NAME, "| 4-bit:", LOAD_4BIT)
print(f"VRAM after judge load: {torch.cuda.memory_allocated()/1e9:.2f} GB") 

## Phase C - external perplexity scoring

Only continuation tokens contribute (the prompt part is masked inside
`_perplexity_core` via labels=-100).

In [ ]:
scores_done = set()
if os.path.exists(SCORES_CSV):
    scores_done = set(pd.read_csv(SCORES_CSV)["id"].tolist())
    print(f"resume: {len(scores_done)} ids already scored")

score_rows, t_score0 = [], time.time()
for rec in tqdm(gen_df.to_dict("records"), total=len(gen_df)):
    if rec["id"] in scores_done:
        continue
    srow = {"id": rec["id"], "topic": rec["topic"]}
    for v in VARIANTS:
        cont = split_continuation(rec["prompt_text"], rec[f"{v}_output"])
        srow[v] = compute_external_perplexity(
            rec["prompt_text"], cont, judge_model, judge_tokenizer, DEVICE)
    score_rows.append(srow)
    if len(score_rows) >= CHUNK_SAVE:
        old = pd.read_csv(SCORES_CSV) if os.path.exists(SCORES_CSV) else None
        merged = pd.concat([old, pd.DataFrame(score_rows)]) if old is not None else pd.DataFrame(score_rows)
        save_chunk(merged, SCORES_CSV)
        score_rows = []
if score_rows:
    old = pd.read_csv(SCORES_CSV) if os.path.exists(SCORES_CSV) else None
    merged = pd.concat([old, pd.DataFrame(score_rows)]) if old is not None else pd.DataFrame(score_rows)
    save_chunk(merged, SCORES_CSV)

t_score = time.time() - t_score0
scores = pd.read_csv(SCORES_CSV)
print(f"scoring done: {len(scores)} rows in {t_score/60:.1f} min") 

## Results

In [ ]:
summary = scores[VARIANTS].agg(["mean", "std"]).T.round(3)
summary["median"] = scores[VARIANTS].median().round(3)
print(summary.to_string())

for v in ["l1_only", "dual"]:
    d = scores[v] - scores["plain"]
    print(f"{v:>8s} vs plain: median paired PPL change {d.median():+.3f} "
          f"({(d / scores['plain']).median() * 100:+.1f}%)")


import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
colors = {"plain": "#8c8c8c", "l1_only": "#1f77b4", "dual": "#2ca02c"}
cap = float(scores[VARIANTS].quantile(0.98).max())
data = [scores[v].clip(upper=cap).values for v in VARIANTS]
ax.violinplot(data, showmedians=True)
ax.set_xticks(range(1, len(VARIANTS) + 1))
labels = [v + "\n(mean " + f"{scores[v].mean():.2f})" for v in VARIANTS]
ax.set_xticklabels(labels)
rng = np.random.default_rng(0)
for i, v in enumerate(VARIANTS):
    ax.scatter(rng.normal(i + 1, 0.05, len(scores)), data[i],
               s=6, alpha=0.35, color=colors[v])
ax.set_ylabel("external PPL (Qwen1.5-7B, continuation-only)")
ax.set_title(f"PPL by watermark variant - C4 rows {ROW_START}-{ROW_START + N_SAMPLES - 1}")
fig.tight_layout()
fig.savefig(f"{FIGDIR}/ppl_violin_99_200.png", dpi=300)
plt.show()

summary.to_csv(f"{OUTDIR}/ppl_summary.csv")
print("timing: generation", round(t_gen / 60, 1), "min | scoring", round(t_score / 60, 1), "min") 